In [ ]:
!pip install --quiet "tacoreader<1.0" omnicloudmask==1.7.0

In [ ]:
REPO_URL = "https://github.com/ArthurrCr/cloudband.git"
PROJECT_DIR = "/content/cloudband"
BRANCH = "main"

import os
import sys

if not os.path.exists(PROJECT_DIR):
    !git clone --quiet {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git fetch --quiet origin && git reset --quiet --hard origin/{BRANCH}

SRC_DIR = f"{PROJECT_DIR}/src"
os.chdir(PROJECT_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

!python -m pytest tests -q

In [ ]:
from pathlib import Path

from cloudband.baselines import ocm
from cloudband.colab.session import reload_package, start
from cloudband.datasets import cloudsen12 as ds
from cloudband.pipelines import phase0

reload_package("cloudband")

In [ ]:
def report(position, total):
    if position % 25 == 0 or position == total:
        print(f"{position}/{total}", flush=True)

In [ ]:
session = start(PROJECT_DIR, require_accelerator=True)

In [ ]:
table = phase0.load_test_split()
print(f"scenes: {len(table)}")
print("pairable:", ds.expected_pairable_scenes(table))

In [ ]:
LIMIT = 5

ocm.check_version()
config = ocm.InferenceConfig(model_version=ocm.LATEST_MODEL_VERSION)

result = phase0.run(
    table,
    lambda stack: ocm.predict_array(stack, config),
    model_id=f"ocm-rgn-published-v{ocm.package_version()}",
    limit=LIMIT,
    progress=report,
)
result.scores

In [ ]:
PIXBOX_S2_V1_7_0 = {"clear": 92.42, "cloud": 91.52, "shadow": 81.37}

print("pairable scenes:", result.pairable)
phase0.compare_to_reference(result, PIXBOX_S2_V1_7_0)

In [ ]:
paths = phase0.save(
    result,
    Path("results/reports"),
    config=config.as_kwargs(),
    package_versions=session.package_versions,
)
for name, path in paths.items():
    print(f"{name}: {path}")